# NYC 311 2025 — Data Cleaning & Validation

This notebook validates the final analytical dataset before geospatial, socioeconomic, SQL, and dashboard analysis.

**Analysis period:** January–December 2025


## 1. Load Final Analytical Dataset



In [1]:
# NYC 311 2025 — DATA CLEANING & VALIDATION
# Notebook 03

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# Clean master analytical dataset
parquet_path = (
    "/content/drive/MyDrive/"
    "NYC_311_2025_BACKUP/"
    "NYC_311_2025_FINAL.parquet"
)

df = pd.read_parquet(parquet_path)

print("NYC 311 2025 — CLEAN DATASET LOADED")
print("=" * 80)
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Memory usage:", f"{df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
print("=" * 80)


Mounted at /content/drive
NYC 311 2025 — CLEAN DATASET LOADED
Rows: 3,655,040
Columns: 19
Memory usage: 3.31 GB


## 2. Dataset Structure Validation



In [2]:
# STEP 2 — FINAL DATASET STRUCTURE VALIDATION

expected_columns = [
    "service_request_id",
    "created_at",
    "closed_at",
    "resolution_hours",
    "resolution_days",
    "resolution_quality",
    "agency",
    "complaint_type",
    "descriptor",
    "status",
    "resolution_description",
    "incident_zip",
    "borough",
    "community_board",
    "community_board_number",
    "community_board_borough",
    "latitude",
    "longitude",
    "geo_quality_final"
]

print("FINAL DATASET STRUCTURE VALIDATION")
print("=" * 80)

print("Expected columns:", len(expected_columns))
print("Actual columns:  ", len(df.columns))

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

unexpected_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("\nMissing expected columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

print("\nColumn order:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2}. {col}")

print("\nVALIDATION RESULT")
print("-" * 80)

if (
    df.columns.tolist() == expected_columns
    and not missing_columns
    and not unexpected_columns
):
    print("PASS — Dataset structure is exactly as expected.")
else:
    print("CHECK REQUIRED — Dataset structure differs from expected.")

print("=" * 80)


FINAL DATASET STRUCTURE VALIDATION
Expected columns: 19
Actual columns:   19

Missing expected columns: []
Unexpected columns: []

Column order:
 1. service_request_id
 2. created_at
 3. closed_at
 4. resolution_hours
 5. resolution_days
 6. resolution_quality
 7. agency
 8. complaint_type
 9. descriptor
10. status
11. resolution_description
12. incident_zip
13. borough
14. community_board
15. community_board_number
16. community_board_borough
17. latitude
18. longitude
19. geo_quality_final

VALIDATION RESULT
--------------------------------------------------------------------------------
PASS — Dataset structure is exactly as expected.


## 3. Service Request ID Validation



In [3]:
# STEP 3 — SERVICE REQUEST ID VALIDATION

print("SERVICE REQUEST ID VALIDATION")
print("=" * 80)

missing_ids = df["service_request_id"].isna().sum()
duplicate_ids = df["service_request_id"].duplicated().sum()
unique_ids = df["service_request_id"].nunique()

print("Total records:", f"{len(df):,}")
print("Unique service request IDs:", f"{unique_ids:,}")
print("Missing service request IDs:", f"{missing_ids:,}")
print("Duplicate service request IDs:", f"{duplicate_ids:,}")

print("\nVALIDATION RESULT")
print("-" * 80)

if (
    missing_ids == 0
    and duplicate_ids == 0
    and unique_ids == len(df)
):
    print("PASS — All service request IDs are unique and non-missing.")
else:
    print("CHECK REQUIRED — ID integrity issue detected.")

print("=" * 80)


SERVICE REQUEST ID VALIDATION
Total records: 3,655,040
Unique service request IDs: 3,655,040
Missing service request IDs: 0
Duplicate service request IDs: 0

VALIDATION RESULT
--------------------------------------------------------------------------------
PASS — All service request IDs are unique and non-missing.


## 4. Date Field Validation



In [4]:
# STEP 4 — DATE FIELD VALIDATION

print("DATE FIELD VALIDATION")
print("=" * 80)

print("CREATED_AT")
print("-" * 80)
print("Data type:", df["created_at"].dtype)
print("Missing:", f"{df['created_at'].isna().sum():,}")
print("Minimum:", df["created_at"].min())
print("Maximum:", df["created_at"].max())

print("\nCLOSED_AT")
print("-" * 80)
print("Data type:", df["closed_at"].dtype)
print("Missing:", f"{df['closed_at'].isna().sum():,}")
print("Minimum:", df["closed_at"].min())
print("Maximum:", df["closed_at"].max())

print("\nDATE RANGE VALIDATION")
print("-" * 80)

created_in_2025 = (
    (df["created_at"] >= "2025-01-01")
    & (df["created_at"] < "2026-01-01")
).all()

print("All created_at values within 2025:", created_in_2025)

print("\nVALIDATION RESULT")
print("-" * 80)

if (
    pd.api.types.is_datetime64_any_dtype(df["created_at"])
    and pd.api.types.is_datetime64_any_dtype(df["closed_at"])
    and df["created_at"].isna().sum() == 0
    and created_in_2025
):
    print("PASS — Date fields are correctly typed and the dataset covers 2025.")
else:
    print("CHECK REQUIRED — Date validation issue detected.")

print("=" * 80)


DATE FIELD VALIDATION
CREATED_AT
--------------------------------------------------------------------------------
Data type: datetime64[ns]
Missing: 0
Minimum: 2025-01-01 00:00:12
Maximum: 2025-12-31 23:59:28

CLOSED_AT
--------------------------------------------------------------------------------
Data type: datetime64[ns]
Missing: 129,036
Minimum: 2024-10-21 11:06:00
Maximum: 2026-09-24 16:02:20

DATE RANGE VALIDATION
--------------------------------------------------------------------------------
All created_at values within 2025: True

VALIDATION RESULT
--------------------------------------------------------------------------------
PASS — Date fields are correctly typed and the dataset covers 2025.


## 5. Resolution Duration Validation



In [5]:
# STEP 5 — RESOLUTION DURATION VALIDATION

print("RESOLUTION DURATION VALIDATION")
print("=" * 80)

negative_hours = (df["resolution_hours"] < 0).sum()
missing_hours = df["resolution_hours"].isna().sum()
zero_hours = (df["resolution_hours"] == 0).sum()

print("Resolution hours — data type:", df["resolution_hours"].dtype)
print("Missing resolution hours:", f"{missing_hours:,}")
print("Negative resolution hours:", f"{negative_hours:,}")
print("Zero-duration records:", f"{zero_hours:,}")

print("\nRESOLUTION QUALITY")
print("-" * 80)

print(
    df["resolution_quality"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nRESOLUTION STATISTICS")
print("-" * 80)

valid_resolution = df["resolution_hours"].dropna()

print("Minimum hours:", valid_resolution.min())
print("Median hours:", valid_resolution.median())
print("Maximum hours:", valid_resolution.max())

print("\nVALIDATION RESULT")
print("-" * 80)

if (
    pd.api.types.is_numeric_dtype(df["resolution_hours"])
    and negative_hours == 0
):
    print("PASS — Resolution duration is numeric and contains no negative values.")
else:
    print("CHECK REQUIRED — Resolution duration issue detected.")

print("=" * 80)


RESOLUTION DURATION VALIDATION
Resolution hours — data type: float64
Missing resolution hours: 129,929
Negative resolution hours: 0
Zero-duration records: 63,068

RESOLUTION QUALITY
--------------------------------------------------------------------------------
resolution_quality
Valid                            3525111
Missing / Invalid Closed Date     129036
Negative Duration                    893

RESOLUTION STATISTICS
--------------------------------------------------------------------------------
Minimum hours: 0.0
Median hours: 6.683333333333334
Maximum hours: 14984.766111111112

VALIDATION RESULT
--------------------------------------------------------------------------------
PASS — Resolution duration is numeric and contains no negative values.


## 6. Categorical Field Validation



In [6]:
# STEP 6 — CATEGORICAL FIELD VALIDATION

print("CATEGORICAL FIELD VALIDATION")
print("=" * 80)

expected_agencies = {
    "New York City Police Department",
    "Department of Housing Preservation and Development",
    "Department of Sanitation",
    "Department of Transportation",
    "Department of Environmental Protection",
    "Department of Parks and Recreation",
    "Department of Buildings",
    "Department of Health and Mental Hygiene",
    "Department of Homeless Services",
    "Taxi and Limousine Commission",
    "Economic Development Corporation",
    "Department of Consumer and Worker Protection",
    "Department of Education",
    "Office of the Sheriff",
    "Office of Technology and Innovation"
}

expected_statuses = {
    "Closed",
    "In Progress",
    "Open",
    "Pending",
    "Assigned",
    "Started"
}

actual_agencies = set(df["agency"].dropna().unique())
actual_statuses = set(df["status"].dropna().unique())

unexpected_agencies = actual_agencies - expected_agencies
missing_agencies = expected_agencies - actual_agencies

unexpected_statuses = actual_statuses - expected_statuses

print("AGENCY")
print("-" * 80)
print("Unique agencies:", df["agency"].nunique())
print("Missing agency values:", f"{df['agency'].isna().sum():,}")
print("Unexpected agency categories:", unexpected_agencies)
print("Missing expected agencies:", missing_agencies)

print("\nCOMPLAINT TYPE")
print("-" * 80)
print("Unique complaint types:", df["complaint_type"].nunique())
print("Missing complaint types:", f"{df['complaint_type'].isna().sum():,}")

print("\nSTATUS")
print("-" * 80)
print("Recognized status records:", f"{df['status'].notna().sum():,}")
print("Missing/unrecognized status records:", f"{df['status'].isna().sum():,}")
print("Unexpected status categories:", unexpected_statuses)

print("\nVALIDATION RESULT")
print("-" * 80)

if (
    df["agency"].isna().sum() == 0
    and not unexpected_agencies
    and not missing_agencies
    and not unexpected_statuses
):
    print("PASS — Standardized categorical fields contain expected categories.")
else:
    print("CHECK REQUIRED — Categorical validation issue detected.")

print("=" * 80)


CATEGORICAL FIELD VALIDATION
AGENCY
--------------------------------------------------------------------------------
Unique agencies: 15
Missing agency values: 0
Unexpected agency categories: set()
Missing expected agencies: set()

COMPLAINT TYPE
--------------------------------------------------------------------------------
Unique complaint types: 874
Missing complaint types: 2,172

STATUS
--------------------------------------------------------------------------------
Recognized status records: 3,504,877
Missing/unrecognized status records: 150,163
Unexpected status categories: set()

VALIDATION RESULT
--------------------------------------------------------------------------------
PASS — Standardized categorical fields contain expected categories.


## 7. Geographic Field Validation



In [7]:
# STEP 7 — GEOGRAPHIC FIELD VALIDATION

print("GEOGRAPHIC FIELD VALIDATION")
print("=" * 80)

expected_boroughs = {
    "BRONX",
    "BROOKLYN",
    "MANHATTAN",
    "QUEENS",
    "STATEN ISLAND"
}

actual_boroughs = set(df["borough"].dropna().unique())
unexpected_boroughs = actual_boroughs - expected_boroughs

valid_latitude = df["latitude"].between(40.4, 41.0, inclusive="both").fillna(False)
valid_longitude = df["longitude"].between(-74.3, -73.6, inclusive="both").fillna(False)

valid_coordinates = (
    df["latitude"].notna()
    & df["longitude"].notna()
)

expected_geo_categories = {
    "Coordinates + Borough + Community Board",
    "Coordinates + Borough",
    "Borough + Community Board",
    "Borough + ZIP",
    "Borough Only",
    "Coordinates Only",
    "No Usable Geography"
}

actual_geo_categories = set(
    df["geo_quality_final"].dropna().unique()
)

unexpected_geo_categories = (
    actual_geo_categories - expected_geo_categories
)

print("BOROUGH")
print("-" * 80)
print("Valid borough records:", f"{df['borough'].notna().sum():,}")
print("Unique boroughs:", df["borough"].nunique())
print("Unexpected borough categories:", unexpected_boroughs)

print("\nCOMMUNITY BOARD")
print("-" * 80)
print("Valid community board records:",
      f"{df['community_board'].notna().sum():,}")
print("Unique community boards:", df["community_board"].nunique())

print("\nCOORDINATES")
print("-" * 80)
print("Valid coordinate pairs:",
      f"{valid_coordinates.sum():,}")
print("Invalid latitude values:",
      f"{(~valid_latitude & df['latitude'].notna()).sum():,}")
print("Invalid longitude values:",
      f"{(~valid_longitude & df['longitude'].notna()).sum():,}")

print("\nZIP CODE")
print("-" * 80)
print("Valid ZIP records:",
      f"{df['incident_zip'].notna().sum():,}")

print("\nGEOGRAPHIC QUALITY")
print("-" * 80)
print("Unexpected geo-quality categories:",
      unexpected_geo_categories)

print("\nVALIDATION RESULT")
print("-" * 80)

if (
    not unexpected_boroughs
    and not unexpected_geo_categories
    and (
        df["latitude"].isna()
        | valid_latitude
    ).all()
    and (
        df["longitude"].isna()
        | valid_longitude
    ).all()
):
    print("PASS — Geographic fields contain valid standardized values.")
else:
    print("CHECK REQUIRED — Geographic validation issue detected.")

print("=" * 80)


GEOGRAPHIC FIELD VALIDATION
BOROUGH
--------------------------------------------------------------------------------
Valid borough records: 3,396,220
Unique boroughs: 5
Unexpected borough categories: set()

COMMUNITY BOARD
--------------------------------------------------------------------------------
Valid community board records: 3,246,595
Unique community boards: 59

COORDINATES
--------------------------------------------------------------------------------
Valid coordinate pairs: 3,303,394
Invalid latitude values: 0
Invalid longitude values: 0

ZIP CODE
--------------------------------------------------------------------------------
Valid ZIP records: 3,476,461

GEOGRAPHIC QUALITY
--------------------------------------------------------------------------------
Unexpected geo-quality categories: set()

VALIDATION RESULT
--------------------------------------------------------------------------------
PASS — Geographic fields contain valid standardized values.


## 8. Geographic Quality Reconciliation



In [8]:
# STEP 8 — GEOGRAPHIC QUALITY RECONCILIATION

print("GEOGRAPHIC QUALITY RECONCILIATION")
print("=" * 80)

geo_counts = (
    df["geo_quality_final"]
    .value_counts(dropna=False)
)

print("GEOGRAPHIC QUALITY DISTRIBUTION")
print("-" * 80)

print(geo_counts.to_string())

print("\nRECONCILIATION")
print("-" * 80)

classified_records = df["geo_quality_final"].notna().sum()
unclassified_records = df["geo_quality_final"].isna().sum()

print("Total records:", f"{len(df):,}")
print("Classified records:", f"{classified_records:,}")
print("Unclassified records:", f"{unclassified_records:,}")

print("\nVALIDATION RESULT")
print("-" * 80)

if (
    classified_records == len(df)
    and unclassified_records == 0
):
    print("PASS — Every record has exactly one geographic-quality classification.")
else:
    print("CHECK REQUIRED — Geographic classification does not fully reconcile.")

print("=" * 80)


GEOGRAPHIC QUALITY RECONCILIATION
GEOGRAPHIC QUALITY DISTRIBUTION
--------------------------------------------------------------------------------
geo_quality_final
Coordinates + Borough + Community Board    3226135
No Usable Geography                         258663
Coordinates + Borough                        77102
Borough Only                                 61248
Borough + Community Board                    20460
Borough + ZIP                                11275
Coordinates Only                               157

RECONCILIATION
--------------------------------------------------------------------------------
Total records: 3,655,040
Classified records: 3,655,040
Unclassified records: 0

VALIDATION RESULT
--------------------------------------------------------------------------------
PASS — Every record has exactly one geographic-quality classification.


## 9. Final Data Quality Validation



In [9]:
# STEP 9 — FINAL DATA QUALITY VALIDATION

print("NYC 311 2025 — FINAL DATA QUALITY VALIDATION")
print("=" * 90)

checks = {}

# 1. Dataset size
checks["Expected row count"] = len(df) == 3_655_040

# 2. Service request IDs
checks["No missing service request IDs"] = (
    df["service_request_id"].isna().sum() == 0
)

checks["No duplicate service request IDs"] = (
    df["service_request_id"].duplicated().sum() == 0
)

# 3. Created dates
checks["No missing created dates"] = (
    df["created_at"].isna().sum() == 0
)

checks["All created dates in 2025"] = (
    (
        (df["created_at"] >= "2025-01-01")
        & (df["created_at"] < "2026-01-01")
    ).all()
)

# 4. Resolution duration
checks["No negative resolution hours"] = (
    (df["resolution_hours"] < 0).sum() == 0
)

checks["Resolution quality fully classified"] = (
    df["resolution_quality"].isna().sum() == 0
)

# 5. Agency
checks["No missing agency values"] = (
    df["agency"].isna().sum() == 0
)

checks["Exactly 15 agencies"] = (
    df["agency"].nunique() == 15
)

# 6. Geographic validity
checks["Exactly 5 boroughs"] = (
    df["borough"].nunique() == 5
)

checks["Exactly 59 community boards"] = (
    df["community_board"].nunique() == 59
)

checks["No invalid latitude values"] = (
    (
        df["latitude"].notna()
        & ~df["latitude"].between(40.4, 41.0)
    ).sum() == 0
)

checks["No invalid longitude values"] = (
    (
        df["longitude"].notna()
        & ~df["longitude"].between(-74.3, -73.6)
    ).sum() == 0
)

# 7. Geographic quality
checks["All records geographically classified"] = (
    df["geo_quality_final"].notna().sum() == len(df)
)

# DISPLAY RESULTS

for check_name, result in checks.items():
    print(f"{'PASS' if result else 'FAIL':6} — {check_name}")

print("\n" + "=" * 90)

overall_pass = all(checks.values())

if overall_pass:
    print("FINAL RESULT: PASS")
    print("The cleaned analytical dataset passed all validation checks.")
else:
    failed_checks = [
        name for name, result in checks.items()
        if not result
    ]

    print("FINAL RESULT: CHECK REQUIRED")
    print("\nFailed checks:")
    for check in failed_checks:
        print(" -", check)

print("=" * 90)


NYC 311 2025 — FINAL DATA QUALITY VALIDATION
PASS   — Expected row count
PASS   — No missing service request IDs
PASS   — No duplicate service request IDs
PASS   — No missing created dates
PASS   — All created dates in 2025
PASS   — No negative resolution hours
PASS   — Resolution quality fully classified
PASS   — No missing agency values
PASS   — Exactly 15 agencies
PASS   — Exactly 5 boroughs
PASS   — Exactly 59 community boards
PASS   — No invalid latitude values
PASS   — No invalid longitude values
PASS   — All records geographically classified

FINAL RESULT: PASS
The cleaned analytical dataset passed all validation checks.


## 10. Cleaning & Validation Methodology



In [10]:
# NYC 311 2025 — DATA CLEANING & VALIDATION METHODOLOGY

methodology = """
NYC 311 2025 — Data Cleaning & Validation Methodology
======================================================

SOURCE
------
NYC 311 Service Requests dataset covering requests created
between January 1, 2025 and December 31, 2025.

DATA INTEGRITY
--------------
• 3,655,040 service requests retained.
• Service request IDs are unique and non-missing.
• No duplicate service request IDs were identified.
• Created timestamps are complete and fall within 2025.

DATE & RESOLUTION HANDLING
---------------------------
• Created and closed timestamps were standardized to datetime.
• Resolution duration was calculated from created and closed timestamps.
• Negative durations were excluded from analytical resolution metrics
  but retained in the resolution quality classification for auditability.
• Missing or invalid closed timestamps were retained as missing.
• Zero-duration requests were retained as valid observations.
• Resolution analysis uses median and percentile statistics because
  resolution times are strongly right-skewed.

CATEGORICAL STANDARDIZATION
---------------------------
• Agency values were standardized into 15 analytical agency categories.
• Complaint types were retained at their source category level without
  unnecessary consolidation.
• Recognized operational statuses were standardized.
• Missing or unrecognized categorical values were retained as missing
  rather than inferred.

GEOGRAPHIC STANDARDIZATION
--------------------------
• Borough values were standardized to the five NYC boroughs.
• Community Board values were validated against the official
  59-board NYC structure.
• Latitude and longitude values were converted to numeric fields and
  restricted to a valid NYC geographic range.
• ZIP codes were standardized and invalid values were treated as missing.
• Geographic quality was classified for every record based on the
  availability of coordinates, borough, Community Board, and ZIP code.

GEOGRAPHIC ANALYSIS RULE
------------------------
Primary spatial analysis will use records with valid boroughs,
validated Community Boards, and/or valid coordinate pairs.

Records with only ZIP information may be used as supplementary
geographic information.

Records without usable geographic information will remain in the
master dataset but will be excluded from spatial analysis.

VALIDATION RESULT
-----------------
The final analytical dataset passed all data-quality validation checks.
"""

print(methodology)



NYC 311 2025 — Data Cleaning & Validation Methodology

SOURCE
------
NYC 311 Service Requests dataset covering requests created
between January 1, 2025 and December 31, 2025.

DATA INTEGRITY
--------------
• 3,655,040 service requests retained.
• Service request IDs are unique and non-missing.
• No duplicate service request IDs were identified.
• Created timestamps are complete and fall within 2025.

DATE & RESOLUTION HANDLING
---------------------------
• Created and closed timestamps were standardized to datetime.
• Resolution duration was calculated from created and closed timestamps.
• Negative durations were excluded from analytical resolution metrics
  but retained in the resolution quality classification for auditability.
• Missing or invalid closed timestamps were retained as missing.
• Zero-duration requests were retained as valid observations.
• Resolution analysis uses median and percentile statistics because
  resolution times are strongly right-skewed.

CATEGORICAL STANDA

## Conclusion

All final dataset quality checks were completed before the downstream analytical stages.
